# Dynamic Partitioning Pruning
- DPP is a technique of pruning partitions at runtime
- Problem Statement: **Analyse the listening activity of users on the release date of a song after `2019-12-31`**

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [3]:
spark_path = "E:\\Spark\\spark-3.2.0-bin-hadoop2.7"

import findspark
findspark.init(spark_path)

In [4]:
from pyspark.storagelevel import StorageLevel
from pyspark.sql.types import *
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

In [5]:
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "10g")
    .master("local[*]")
    .appName("Dynamic Partition Pruning")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")

E:\Spark\spark-3.2.0-bin-hadoop2.7\python\pyspark\context.py:238: FutureWarning: Python 3.6 support is deprecated in Spark 3.2.
  FutureWarning


In [6]:
df_listening_actv = spark.read.csv("../data/partitioning/raw/Spotify_Listening_Activity.csv", header=True, inferSchema=True)

df_listening_actv = (
    df_listening_actv
    .withColumnRenamed("listen_date", "listen_time")
    .withColumn("listen_date", F.to_date("listen_time", "yyyy-MM-dd HH:mm:ss.SSSSSS"))
)

# Partitioning listening activity by the listen date
(
    df_listening_actv
    .write
    .partitionBy("listen_date")
    .mode("overwrite")
    .parquet("./output/listening_activity_pt")
)

In [7]:
df_listening_actv_pt = spark.read.parquet("./output/listening_activity_pt")
df_listening_actv_pt.show(5, False)

+-----------+-------+--------------------------+---------------+-----------+
|activity_id|song_id|listen_time               |listen_duration|listen_date|
+-----------+-------+--------------------------+---------------+-----------+
|4456       |16     |2023-07-18 10:15:47.023264|151            |2023-07-18 |
|4457       |65     |2023-07-18 10:15:47.023264|181            |2023-07-18 |
|4458       |60     |2023-07-18 10:15:47.023264|280            |2023-07-18 |
|4459       |3      |2023-07-18 10:15:47.023264|249            |2023-07-18 |
|4460       |45     |2023-07-18 10:15:47.023264|130            |2023-07-18 |
+-----------+-------+--------------------------+---------------+-----------+
only showing top 5 rows



In [8]:
df_songs = spark.read.csv("../data/partitioning/raw/Spotify_Songs.csv", header=True, inferSchema=True)
df_songs.printSchema()

root
 |-- song_id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- artist_id: integer (nullable = true)
 |-- release_date: string (nullable = true)



In [9]:
df_songs = (
    df_songs
    .withColumnRenamed("release_date", "release_datetime")
    .withColumn("release_date", F.to_date("release_datetime", "yyyy-MM-dd HH:mm:ss.SSSSSS"))
)
df_songs.show(5, False)
df_songs.printSchema()

+-------+------+---------+--------------------------+------------+
|song_id|title |artist_id|release_datetime          |release_date|
+-------+------+---------+--------------------------+------------+
|1      |Song_1|2        |2021-10-15 10:15:47.006571|2021-10-15  |
|2      |Song_2|45       |2020-12-07 10:15:47.006588|2020-12-07  |
|3      |Song_3|25       |2022-07-11 10:15:47.006591|2022-07-11  |
|4      |Song_4|25       |2019-03-09 10:15:47.006593|2019-03-09  |
|5      |Song_5|26       |2019-09-07 10:15:47.006596|2019-09-07  |
+-------+------+---------+--------------------------+------------+
only showing top 5 rows

root
 |-- song_id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- artist_id: integer (nullable = true)
 |-- release_datetime: string (nullable = true)
 |-- release_date: date (nullable = true)



In [11]:
# Pick songs released in 2020
df_selected_songs = df_songs.filter(F.col("release_date") > F.lit("2019-12-31"))

In [14]:
df_listening_actv_of_selected_songs = ( 
    df_listening_actv_pt
    .join(
        df_selected_songs, 
        (df_songs.release_date == df_listening_actv_pt.listen_date) & (df_songs.song_id == df_listening_actv_pt.song_id), 
        "inner"
    )
)

In [15]:
df_listening_actv_of_selected_songs.explain(True)

== Parsed Logical Plan ==
Join Inner, ((release_date#104 = listen_date#48) AND (song_id#91 = song_id#45))
:- Relation [activity_id#44,song_id#45,listen_time#46,listen_duration#47,listen_date#48] parquet
+- Filter (release_date#104 > cast(2019-12-31 as date))
   +- Project [song_id#91, title#92, artist_id#93, release_datetime#99, to_date('release_datetime, Some(yyyy-MM-dd HH:mm:ss.SSSSSS)) AS release_date#104]
      +- Project [song_id#91, title#92, artist_id#93, release_date#94 AS release_datetime#99]
         +- Relation [song_id#91,title#92,artist_id#93,release_date#94] csv

== Analyzed Logical Plan ==
activity_id: int, song_id: int, listen_time: string, listen_duration: int, listen_date: date, song_id: int, title: string, artist_id: int, release_datetime: string, release_date: date
Join Inner, ((release_date#104 = listen_date#48) AND (song_id#91 = song_id#45))
:- Relation [activity_id#44,song_id#45,listen_time#46,listen_duration#47,listen_date#48] parquet
+- Filter (release_date#104

In [16]:
df_listening_actv_of_selected_songs.show()

+-----------+-------+--------------------+---------------+-----------+-------+-------+---------+--------------------+------------+
|activity_id|song_id|         listen_time|listen_duration|listen_date|song_id|  title|artist_id|    release_datetime|release_date|
+-----------+-------+--------------------+---------------+-----------+-------+-------+---------+--------------------+------------+
|       9760|     89|2023-07-24 10:15:...|             81| 2023-07-24|     89|Song_89|       33|2023-07-24 10:15:...|  2023-07-24|
|       9768|     89|2023-07-24 10:15:...|            295| 2023-07-24|     89|Song_89|       33|2023-07-24 10:15:...|  2023-07-24|
|       9799|     89|2023-07-24 10:15:...|            272| 2023-07-24|     89|Song_89|       33|2023-07-24 10:15:...|  2023-07-24|
|       7322|     64|2023-10-25 10:15:...|             95| 2023-10-25|     64|Song_64|       32|2023-10-25 10:15:...|  2023-10-25|
+-----------+-------+--------------------+---------------+-----------+-------+-----

In [17]:
spark.stop()